In [ ]:
#!pip install pandas-datareader


  Using cached pandas_datareader-0.11.1-py3-none-any.whl.metadata (3.8 kB)
Using cached pandas_datareader-0.11.1-py3-none-any.whl (65 kB)


In [ ]:

# downloads UNRATE, FEDFUNDS, CPIAUCSL, and GDP,
# handles the structural upsampling, calculates the Year-over-Year inflation rate percentage,
# and streams the aligned monthly sheets straight to S3 folder path [13/Oct/2026]

import io
import datetime
import boto3
import pandas as pd
import pandas_datareader.data as web

# TARGET AWS STORAGE CONFIGURATION
TARGET_S3_BUCKET = "TARGET_S3_BUCKET"
TARGET_S3_PREFIX = "TARGET_S3_BUCKET/"

s3_client = boto3.client("s3", region_name="us-west-2")

def harvest_and_sync_gdp_inflation_to_s3():
    # Set lookback bounds from January 1, 1980
    start_date = datetime.date(1980, 1, 1)
    
    # Dynamic Calendar Calculation: Locks end boundary to the final day of last month
    today = datetime.date.today()
    first_day_of_this_month = today.replace(day=1)
    end_date = first_day_of_this_month - datetime.timedelta(days=1)
    
    # CORE DATA CODES FROM FRED:
    # UNRATE: Unemployment Rate (Monthly)
    # FEDFUNDS: Effective Federal Funds Rate (Monthly)
    # CPIAUCSL: Consumer Price Index (Monthly - used to calculate Inflation YoY)
    # GDP: Gross Domestic Product (Quarterly - requires upsampling)
    fred_codes = ["UNRATE", "FEDFUNDS", "CPIAUCSL", "GDP"]
    
    print("=" * 80)
    print(f" Ingesting GDP & Inflation Rate Matrix [Since 1980]")
    print(f" Lookback Target End: {end_date}")
    print("=" * 80)
    
    try:
        print(f" Querying over-the-air FRED API layers for {fred_codes}...")
        df_raw = web.DataReader(fred_codes, "fred", start_date, end_date)
        
        # FEATURE 1: Upsample Quarterly GDP to Monthly via Forward-Filling
        # This matches the structural dimension of the monthly index rows perfectly
        df_monthly = df_raw.resample('MS').first() # Resample to Month Start
        df_monthly['GDP'] = df_monthly['GDP'].ffill()
        
        # FEATURE 2: Transform raw CPI points into a true Year-over-Year (YoY) Inflation Rate %
        df_monthly['INFLATION'] = df_monthly['CPIAUCSL'].pct_change(periods=12) * 100
        
        # Isolate our target 4-feature dataframe and drop initialization NaN rows
        df_final = df_monthly[["UNRATE", "FEDFUNDS", "GDP", "INFLATION"]].dropna()
        print(f" Success! Aligned multi-variant dataframe rows captured: {len(df_final)}")
        
    except Exception as e:
        print(f" Data Reader extraction pipeline failed: {e}")
        return

    # Loop through the final engineered vectors and stream independent sheets to S3
    for series_id in ["UNRATE", "FEDFUNDS", "GDP", "INFLATION"]:
        target_s3_key = f"{TARGET_S3_PREFIX}fred_{series_id.lower()}.csv"
        
        # Structure clean two-column flat text CSV documents
        csv_buffer = "date,value\n"
        for timestamp, row in df_final.iterrows():
            date_str = timestamp.strftime("%Y-%m-%d")
            metric_val = float(row[series_id])
            csv_buffer += f"{date_str},{metric_val}\n"
            
        csv_bytes_stream = io.BytesIO(csv_buffer.encode('utf-8'))
        
        print(f" Streaming matrix file directly into path: s3://{TARGET_S3_BUCKET}/{target_s3_key}")
        try:
            s3_client.upload_fileobj(
                Fileobj=csv_bytes_stream,
                Bucket=TARGET_S3_BUCKET,
                Key=target_s3_key
            )
            print(f" SUCCESS! Amazon S3 mirror asset updated: s3://{TARGET_S3_BUCKET}/{target_s3_key}")
        except Exception as e:
            print(f" Failed to stream object payload to S3 Storage layer: {e}")
            
    print("=" * 80 + "\n Data pipeline completely synchronized with GDP and Inflation vectors!")

# Fire the dynamic data sync
harvest_and_sync_gdp_inflation_to_s3()

📡 Ingesting GDP & Inflation Rate Matrix [Since 1980]
   • Lookback Target End: 2026-09-30
📡 Querying over-the-air FRED API layers for ['UNRATE', 'FEDFUNDS', 'CPIAUCSL', 'GDP']...
✅ Success! Aligned multi-variant dataframe rows captured: 548
📤 Streaming matrix file directly into path: s3://us-recession-model-bucket/data/fred_unrate.csv


/tmp/ipykernel_386/3142063358.py:48: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_monthly['INFLATION'] = df_monthly['CPIAUCSL'].pct_change(periods=12) * 100


🎉 SUCCESS! Amazon S3 mirror asset updated: s3://us-recession-model-bucket/data/fred_unrate.csv
📤 Streaming matrix file directly into path: s3://us-recession-model-bucket/data/fred_fedfunds.csv
🎉 SUCCESS! Amazon S3 mirror asset updated: s3://us-recession-model-bucket/data/fred_fedfunds.csv
📤 Streaming matrix file directly into path: s3://us-recession-model-bucket/data/fred_gdp.csv
🎉 SUCCESS! Amazon S3 mirror asset updated: s3://us-recession-model-bucket/data/fred_gdp.csv
📤 Streaming matrix file directly into path: s3://us-recession-model-bucket/data/fred_inflation.csv
🎉 SUCCESS! Amazon S3 mirror asset updated: s3://us-recession-model-bucket/data/fred_inflation.csv
🚀 Data pipeline completely synchronized with GDP and Inflation vectors!
